# Exploratory Data Analysis — IEEE-CIS Fraud Detection

IEEE-CIS Fraud Detection veri setinin ilk keşifsel analizi: veri boyutu, eksik değer yapısı, hedef değişken (`isFraud`) dağılımı, işlem tutarı ve zaman aralığı.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

## 1. Veri Yükleme ve Birleştirme

`transaction` ve `identity` tabloları `TransactionID` üzerinden left join ile birleştirilir; her işlemin bir identity (cihaz/tarayıcı) kaydı bulunmayabilir.

In [ ]:
df_transaction = pd.read_csv("../data/train_transaction.csv")
df_identity = pd.read_csv("../data/train_identity.csv")
df = pd.merge(df_transaction, df_identity, on="TransactionID", how="left")

## 2. Genel Bakış

In [ ]:
print(df.shape)

In [ ]:
df.info()

In [ ]:
df.head()

## 3. Eksik Değer Analizi

In [ ]:
missing_pct = df.isna().mean()
missing_pct.sort_values(ascending=False).head(20)

Eksik değer oranları dört aralıkta gruplandırılmıştır:

In [ ]:
missing_buckets = pd.cut(
    missing_pct,
    bins=[0.0, 0.1, 0.5, 0.9, 1.0],
    labels=["%0-10", "%10-50", "%50-90", "%90-100"],
    include_lowest=True,
)
missing_buckets.value_counts()

**Bulgu:** 434 sütunun ~%47'si (202 sütun) `%50-90` aralığında eksik değere sahip; yalnızca 12 sütun `%90`'ın üzerinde eksik. Bu dağılım, feature engineering aşamasında hangi ham sütunların doğrudan kullanılacağına, hangilerinin bir "eksik mi" bayrağıyla temsil edileceğine karar verirken referans alınacaktır.

## 4. Hedef Değişken (`isFraud`) Dağılımı

In [ ]:
fraud_class = df["isFraud"].value_counts(normalize=True)
fraud_class

In [ ]:
sns.countplot(data=df, x="isFraud")

**Bulgu:** İşlemlerin %96.5'i normal, %3.5'i dolandırıcılık — ciddi bir sınıf dengesizliği söz konusu. Bu nedenle model değerlendirmesinde accuracy yerine PR-AUC esas alınacak; sınıf dengesizliği SMOTE ile yapay örnekleme yerine class weight (maliyet-duyarlı öğrenme) ile ele alınacaktır.

## 5. TransactionAmt ve Zaman (TransactionDT) Analizi

İşlem tutarının (`TransactionAmt`) fraud ve normal işlemler arasındaki dağılımı ile veri setinin kapsadığı zaman aralığı (`TransactionDT`) incelenmiştir.

In [ ]:
df.groupby("isFraud")["TransactionAmt"].describe()

In [ ]:
sns.boxplot(data=df, x="isFraud", y="TransactionAmt")
plt.yscale("log")

**Bulgu:** Fraud işlemlerin ortalama (149.2) ve medyan (75.0) tutarları normal işlemlerden (sırasıyla 134.5 ve 68.5) biraz daha yüksek; ancak asıl belirgin fark dağılımın genişliğinde — fraud işlemlerin orta %50'lik aralığı (IQR ≈ 126) normal işlemlerin neredeyse iki katı (IQR ≈ 76), yani fraud işlemler tutar açısından daha öngörülemez. İki dağılım büyük ölçüde çakıştığından ham `TransactionAmt` tek başına güçlü bir ayırt edici değil; kullanıcının kendi harcama geçmişinden sapmayı ölçen türetilmiş özellikler (`amount_deviation_from_user`) daha değerli olacaktır.

In [ ]:
transaction_dt_min_days = df["TransactionDT"].min() / (60 * 60 * 24)
transaction_dt_max_days = df["TransactionDT"].max() / (60 * 60 * 24)
print(transaction_dt_min_days, transaction_dt_max_days)

**Bulgu:** `TransactionDT`, gerçek bir takvim tarihi değil, bir referans noktasından itibaren geçen saniye sayısıdır. Veri seti yaklaşık 183 günlük (~6 aylık) bir dönemi kapsamaktadır. Bu, model geliştirme aşamasında uygulanacak zaman bazlı (temporal) train/test ayrımı için veri setinin toplam süresini netleştirir — örneğin ilk ~5 ay train, son ~1 ay test olarak kullanılabilir; rastgele bölme, gelecekteki bilgiyi geçmişi tahmin etmekte kullanma riski (zaman sızıntısı) taşır.